# Guitarra polifónica: GuitarSet + Demucs
Adjuntá [alonhaviv/guitarset](https://www.kaggle.com/datasets/alonhaviv/guitarset) mediante **Add Input**. Activá GPU e Internet para clonar el código y descargar los pesos preentrenados de Demucs. El audio de GuitarSet se lee directamente desde `/kaggle/input/guitarset`: no se descarga ni se copia. Los checkpoints y la caché se escriben en `/kaggle/working`, que [Kaggle permite reutilizar como entrada de otra versión](https://www.kaggle.com/docs/notebooks).

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
REPO = Path('/kaggle/working/mia-final-project')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Kajachuan/mia-final-project.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## Configuración de la sesión
En una sesión posterior, adjuntá la salida de la versión anterior mediante **Add Input → Notebook Output Files** y escribí su ruta en `PREVIOUS_OUTPUT`. `TOTAL_EPOCHS` es el total acumulado deseado, no la cantidad adicional. El presupuesto de tiempo detiene el entrenamiento de forma limpia para que puedas guardar una versión antes del límite de sesión.

In [ ]:
DATASET_ROOT = Path('/kaggle/input/guitarset')
PREVIOUS_OUTPUT = None  # Ejemplo: '/kaggle/input/mi-notebook-anterior'
TOTAL_EPOCHS = 12
RUN_BUDGET_MINUTES = 180
CHECKPOINT_EVERY_BATCHES = 25
WORK = Path('/kaggle/working')
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)
previous = Path(PREVIOUS_OUTPUT) if PREVIOUS_OUTPUT else None
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError('Adjuntá alonhaviv/guitarset con Add Input')
if previous and not previous.is_dir():
    raise FileNotFoundError(previous)
STARTED = time.monotonic()
CACHE = previous / 'guitarset/cache' if previous and (previous / 'guitarset/cache/manifest.json').exists() else WORK / 'guitarset/cache'
print('Dataset:', DATASET_ROOT, '| Caché:', CACHE, '| Salida:', RUNS)

In [ ]:
from polytranscribe.features import FEATURE_VERSION
manifest_path = CACHE / 'manifest.json'
compatible = manifest_path.exists() and json.loads(manifest_path.read_text()).get('feature_version') == FEATURE_VERSION
if not compatible:
    subprocess.run([sys.executable, '-m', 'polytranscribe.prepare',
                    '--source-root', str(DATASET_ROOT),
                    '--root', str(WORK / 'guitarset')], check=True)
    CACHE = WORK / 'guitarset/cache'
print('Caché lista:', CACHE)

In [ ]:
def resume_checkpoint(variant):
    candidates = [RUNS / f'{variant}_last.pt']
    if previous:
        candidates.append(previous / 'runs' / f'{variant}_last.pt')
    return next((path for path in candidates if path.exists()), None)

deadline = STARTED + RUN_BUDGET_MINUTES * 60
for variant in ('baseline', 'gated'):
    remaining = (deadline - time.monotonic()) / 60 - 5  # reservar 5 min para guardar la versión
    if remaining <= 0:
        print('Presupuesto agotado; guardá esta versión y continuá en otra sesión')
        break
    command = [sys.executable, '-m', 'polytranscribe.train', '--cache', str(CACHE),
               '--output', str(RUNS), '--variant', variant, '--epochs', str(TOTAL_EPOCHS),
               '--checkpoint-every', str(CHECKPOINT_EVERY_BATCHES),
               '--max-runtime-minutes', str(remaining)]
    resume = resume_checkpoint(variant)
    if resume:
        command += ['--resume', str(resume)]
    print('Entrenando:', variant, '| resume:', resume, flush=True)
    subprocess.run(command, check=True)
    state = json.loads((RUNS / f'{variant}_status.json').read_text())
    print('Estado:', state, flush=True)
    if state['status'] != 'complete':
        print('Guardá esta versión y adjuntá su salida en la próxima sesión')
        break

In [ ]:
for variant in ('baseline', 'paper_gate', 'gated'):
    result_path = RUNS / f'{variant}.json'
    if result_path.exists():
        result = json.loads(result_path.read_text())
        print(variant, 'frame F1:', result['test']['frame']['f1'],
              'polyphonic frame F1:', result['test']['polyphonic_frame']['f1'],
              'note F1:', result['test']['note']['f1'],
              'onset+pitch F1:', result['test']['onset_note']['f1'])
print('Checkpoints reanudables:', sorted(RUNS.glob('*_last.pt')))

## Evaluación cuantitativa opcional en mezclas
Adjuntá [BabySlakh](https://zenodo.org/records/4603870) extraído como entrada y asigná su carpeta a `SLAKH_ROOT`. Sus 20 temas sintéticos sirven para verificar el protocolo; para resultados finales usá el test de Slakh2100-redux. El evaluador compara MIDI de guitarra ideal y guitarra separada. `WITH_SI_SDR` añade únicamente un diagnóstico de separación.

In [ ]:
SLAKH_ROOT = None  # Ejemplo: '/kaggle/input/babyslakh/babyslakh_16k'
WITH_SI_SDR = False
eval_status = RUNS / 'gated_status.json'
eval_ready = eval_status.exists() and json.loads(eval_status.read_text())['status'] == 'complete'
if SLAKH_ROOT and eval_ready and time.monotonic() < deadline - 60:
    command = [sys.executable, '-m', 'polytranscribe.eval_slakh',
               '--data-root', str(Path(SLAKH_ROOT)), '--checkpoint', str(RUNS / 'gated.pt'),
               '--output', str(RUNS / 'slakh_eval'), '--split', 'any',
               '--max-tracks', '3', '--seconds', '30']
    if WITH_SI_SDR:
        command.append('--with-si-sdr')
    subprocess.run(command, check=True)
else:
    print('Evaluación Slakh omitida: asigná SLAKH_ROOT después de adjuntar el dataset')

## Prueba funcional de mezcla → Demucs → `guitar` → MIDI
Asigná `MIXTURE_AUDIO` si adjuntaste una mezcla propia. De lo contrario se usa una guitarra aislada solo para verificar que el flujo se ejecute; eso no mide calidad de separación.

In [ ]:
MIXTURE_AUDIO = None  # Ejemplo: '/kaggle/input/mi-mezcla/cancion.wav'
gated_status = RUNS / 'gated_status.json'
ready = gated_status.exists() and json.loads(gated_status.read_text())['status'] == 'complete'
if ready and time.monotonic() < deadline - 60:
    demo_audio = Path(MIXTURE_AUDIO) if MIXTURE_AUDIO else next(DATASET_ROOT.rglob('*_mic.wav'))
    print('Entrada de prueba:', demo_audio)
    subprocess.run([sys.executable, '-m', 'polytranscribe.pipeline', '--audio', str(demo_audio),
                    '--checkpoint', str(RUNS / 'gated.pt'),
                    '--output-dir', str(RUNS / 'pipeline_demo')], check=True)
else:
    print('Se omite la prueba funcional para reservar tiempo o porque falta completar gated')

## Continuar otro día
Usá **Save Version** para conservar `/kaggle/working`. En otra sesión, adjuntá esa salida como **Notebook Output Files**, poné su ruta en `PREVIOUS_OUTPUT` y ejecutá el notebook. El archivo `<variante>_last.pt` guarda modelo, optimizador, RNG, época y próximo lote; `--epochs` indica el total de épocas buscado. Se guarda cada 25 lotes y al cerrar una época. Si el proceso termina abruptamente, se repiten como máximo los lotes posteriores al último checkpoint disponible; parar antes del límite con `RUN_BUDGET_MINUTES` evita depender de un cierre abrupto.